# LONDON DEMOGRAPHIC ANALYSIS BY AREA

In [1]:
# import libraries
from IPython.display import display, HTML
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import matplotlib.animation as animation
from sklearn.neighbors import KDTree
import re
import contextily as ctx
from scipy.stats import spearmanr
import os
import time
from tqdm.notebook import tqdm
import sys
sys.path.append('..')
from functions.plotting_functions import plot_london, highlight_area, make_interactive_map

# set default font for plots
plt.rcParams["font.family"] = "Times New Roman"

In [ ]:
base_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
data_dir = os.path.join(base_dir, 'data')

borough_outline_filepath = os.path.join(data_dir,'processed','ldn_borough_outline.geojson')
lsoa_outline_filepath = os.path.join(data_dir,'processed','ldn_LSOA_outline.geojson')

borough_outline = gpd.read_file(borough_outline_filepath)
lsoa_outline = gpd.read_file(lsoa_outline_filepath)

age_sex_filepath = os.path.join(data_dir,'processed','ons_age_sex_data_2022.csv')
age_sex = pd.read_csv(age_sex_filepath)
age_sex = age_sex[['LSOA21CD', 'BOROUGH', 'OUTCODE', 'Total'] + 
                  age_sex.columns[age_sex.columns.str.contains('F')|age_sex.columns.str.contains('M')].tolist()]

KeyboardInterrupt: 

In [ ]:
# separate sex
age_sex['F'] = age_sex.loc[:,age_sex.columns.str.contains('F')].sum(axis=1)
age_sex['M'] = age_sex.loc[:,age_sex.columns.str.contains('M')].sum(axis=1)
age_sex['F%'] = (age_sex['F']/age_sex['Total'])*100

# separate ages
for age in range(100):
    female = 'F'+str(age)
    male = 'M'+str(age)
    try:
        age_sex[str(age)] = age_sex[[female, male]].sum(axis=1)
    except:
        continue

age_cols = np.linspace(0, 90, 91).astype('int').astype('str')

age_sex['MODE_AGE'] = age_sex.loc[:, age_sex.columns.str.isnumeric()].idxmax(axis='columns').astype('int')
age_sex

In [ ]:
age_sex_geometry = gpd.GeoDataFrame(pd.merge(age_sex, lsoa_outline.drop(columns=['BOROUGH', 'OUTCODE']), on='LSOA21CD', how='right'),
                                        geometry='geometry')

In [ ]:
# split of men and women across London
colours = ["#032895", '#FFFFFF', "#C70A68"]  # Pink -> White -> Blue

n_bins = 11 #256 #7
Bu_Pk = mcolors.LinearSegmentedColormap.from_list('pink_blue', colours, N=n_bins)

plt.figure(figsize=(12,7))
plot_london(age_sex_geometry, col='F%', cmap=Bu_Pk, leg_label='Percentage Female (%)')
plot_london(borough_outline, borough=True, edgecolour='black')
plt.title('Percentage of men and women across London', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
n_bins = 9
inferno_bin = mpl.colormaps['inferno_r'].resampled(n_bins)
plt.figure(figsize=(12,7))
plot_london(age_sex_geometry, col='MODE_AGE', cmap=inferno_bin, leg_label='Age')
plot_london(borough_outline, borough=True, edgecolour='black')
plt.title('Most Frequent Age Group by London Region', fontsize=16)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,7))
plot_london(age_sex_geometry, col='MODE_AGE', cmap='Set1', leg_label='Age')
plot_london(borough_outline, borough=True, edgecolour='black')
plt.title('Most Frequent Age Group by London Region', fontsize=15)
plt.tight_layout()
plt.show()